# Download DeepImage (Deep10M) Dataset

This notebook downloads the **DeepImage (96-dimension, 10 million vectors)** dataset directly from the official `ann-benchmarks` repository. 

It is already pre-packaged in an `.hdf5` format. The notebook will automatically download the base dataset (`train`) and the official evaluation queries (`test`), and it will programmatically extract 10,000 vectors to act as the `learn` split for training your predictors!

**Instructions:**
1. Connect to any Google Colab runtime (CPU or GPU, it doesn't matter).
2. Run the cells to mount your Google Drive and download the file.
3. The 3.8 GB file will be saved securely to your Google Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Download the DeepImage dataset to Colab's LOCAL storage first
# Google Drive sync can cause 'truncated file' errors if we try to modify a massive file directly on the drive.
import os
import shutil

local_file = "/content/deep-image-96-angular.hdf5"
drive_file = "/content/drive/MyDrive/deep-image-96-angular.hdf5"

print(f"Downloading DeepImage dataset to {local_file}...")
!wget -c -O "{local_file}" http://ann-benchmarks.com/deep-image-96-angular.hdf5
print("\nDownload complete!")

In [ ]:
# 3. Create the 'Learn' Split (10,000 vectors for calibration)
# We do this on the local fast storage to prevent Google Drive corruption
import h5py
import numpy as np

print("Generating 'learn' split...")
with h5py.File(local_file, 'a') as f:
    if 'learn' not in f:
        # Randomly sample 10,000 indices from the base corpus
        np.random.seed(42)
        num_base = f['train'].shape[0]
        learn_indices = np.random.choice(num_base, 10000, replace=False)
        
        # Extract and save under 'learn' key
        learn_vectors = f['train'][np.sort(learn_indices)]
        f.create_dataset('learn', data=learn_vectors)
        print("Successfully created 10,000 'learn' vectors!")
    else:
        print("'learn' split already exists.")

In [ ]:
# 4. Verify all splits locally
print("\nVerifying dataset splits...")
with h5py.File(local_file, 'r') as f:
    print(f"Base dataset ('train'): {f['train'].shape}")
    print(f"Test Queries ('test'):  {f['test'].shape}")
    print(f"Learn Queries ('learn'): {f['learn'].shape}")
    print("\nFile is fully intact!")

In [ ]:
# 5. Copy the finalized, safe file to your Google Drive
print(f"\nCopying final file to your Google Drive at {drive_file}...")
!cp "{local_file}" "{drive_file}"
print("Done! You are ready to benchmark.")